# Klasik Görüntü İşleme ile Hücre Segmentasyonu, Sayımı ve Zamansal Analizi

## Bölüm 2 – Veri Setinin İndirilmesi ve Ön İncelemesi

In [ ]:
# Bölüm 2.1–2.2: Veri seti klasörlerini ve sekansları inceleme
from pathlib import Path

VERI = Path("data/Fluo-N2DH-GOWT1")

for klasor in sorted(VERI.iterdir()):
    tif_sayisi = len(list(klasor.rglob("*.tif")))
    print(f"{klasor.name:12s} -> {tif_sayisi} tif dosyası")

In [ ]:
# Bölüm 2.3: Görüntülerin boyutu, veri tipi ve piksel yoğunlukları
#
# Test sızıntısını önlemek için:
#   - Sekans 01 (geliştirme + doğrulama havuzu): boyut, tip ve yoğunluklar detaylı incelenir.
#   - Sekans 02 (test havuzu): yalnızca kare sayısı, boyut ve veri tipi kontrol edilir.
#     Yoğunluk/histogram bilgisine bakılmaz; yöntem seçimini etkilememesi için.
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

def goruntu_oku(yol):
    """TIFF dosyasını orijinal bit derinliğiyle okur.
    cv2.imread Türkçe karakterli yollarda çalışmadığı için
    dosya önce bayt olarak okunur, sonra cv2.imdecode ile çözülür."""
    veri = np.fromfile(str(yol), dtype=np.uint8)
    return cv2.imdecode(veri, cv2.IMREAD_UNCHANGED)

# --- Sekans 02: yalnızca biçim kontrolü ---
dosyalar_02 = sorted((VERI / "02").glob("t*.tif"))
bicimler_02 = {(goruntu_oku(d).shape, str(goruntu_oku(d).dtype)) for d in dosyalar_02}
print(f"Sekans 02: {len(dosyalar_02)} kare, (boyut, tip) = {bicimler_02}")

# --- Sekans 01: detaylı inceleme ---
satirlar = []
for dosya in sorted((VERI / "01").glob("t*.tif")):
    g = goruntu_oku(dosya)
    satirlar.append({
        "kare": int(dosya.stem[1:]),          # "t042" -> 42
        "boyut": f"{g.shape[0]}x{g.shape[1]}",
        "tip": str(g.dtype),
        "min": int(g.min()),
        "max": int(g.max()),
        "ortalama": g.mean(),
        "p99": np.percentile(g, 99),          # piksellerin %99'u bu değerin altında
        "sifir_orani": (g == 0).mean(),
    })

istatistik = pd.DataFrame(satirlar)
print(f"Sekans 01: {len(istatistik)} kare, boyutlar = {istatistik.boyut.unique()}, tipler = {istatistik.tip.unique()}")

# Sekans 01 sayısal özeti (kareler arası dağılım)
istatistik[["min", "max", "ortalama", "p99", "sifir_orani"]].describe().round(3)

In [ ]:
# Bölüm 2.3: Histogram özellikleri - Sekans 01'in TÜM karelerinin birleşik histogramı
toplam = np.zeros(256)
for dosya in sorted((VERI / "01").glob("t*.tif")):
    toplam += np.bincount(goruntu_oku(dosya).ravel(), minlength=256)
hist_01 = toplam / toplam.sum()               # oran (0-1) olarak normalize

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.arange(256), hist_01, label="Sekans 01 (92 kare)")
ax.set_yscale("log")                           # arka plan tepesi diğer değerleri ezmesin
ax.set_xlabel("Piksel yoğunluğu")
ax.set_ylabel("Piksel oranı (log)")
ax.set_title("Sekans 01 - tüm karelerin birleşik histogramı")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

# Sayısal özet: yoğunluğun hangi aralıkta toplandığı
kum = np.cumsum(hist_01)
for oran in [0.50, 0.90, 0.99, 0.999]:
    print(f"Piksellerin %{oran*100:g}'i <= {np.searchsorted(kum, oran)}")

In [ ]:
# Bölüm 2.3: Sekans 01'de parlaklığın kare kare (zamanla) değişimi
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(istatistik.kare, istatistik.ortalama)
ax[1].plot(istatistik.kare, istatistik.p99)
ax[0].set_title("Sekans 01 - kare ortalama yoğunluğu")
ax[1].set_title("Sekans 01 - kare 99. yüzdelik yoğunluğu (parlak bölgeler)")
for a in ax:
    a.set_xlabel("Kare numarası")
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Bölüm 2.4: Hücrelerin şekil, boyut ve parlaklık farklılıkları
#
# 01_GT/SEG klasöründeki manuel segmentasyon maskeleri hücre sınırlarını hassas biçimde verir.
# (Bu maskeler her karede tüm hücreleri değil, yalnızca bir kısmını içerir; ama ölçüm için yeterli.)
# Her etiketli hücre için alan, eşdeğer çap, şekil ve ortalama parlaklık ölçülür.
from skimage.measure import regionprops_table

SEG_KLASOR = VERI / "01_GT" / "SEG"
hucre_satirlari = []
for maske_yolu in sorted(SEG_KLASOR.glob("man_seg*.tif")):
    kare_no = int(maske_yolu.stem.replace("man_seg", ""))     # "man_seg014" -> 14
    maske = goruntu_oku(maske_yolu)                            # her hücre farklı bir etiket (1, 2, ...)
    goruntu = goruntu_oku(VERI / "01" / f"t{kare_no:03d}.tif")
    olcum = regionprops_table(
        maske, intensity_image=goruntu,
        properties=["label", "area", "equivalent_diameter_area", "eccentricity",
                    "solidity", "intensity_mean", "intensity_min", "intensity_max"])
    tablo = pd.DataFrame(olcum)
    tablo["kare"] = kare_no
    hucre_satirlari.append(tablo)

hucreler = pd.concat(hucre_satirlari, ignore_index=True)
print(f"{hucreler.kare.nunique()} SEG karesinden toplam {len(hucreler)} hücre ölçüldü.\n")
print("eccentricity: 0 = tam daire, 1'e yaklaştıkça uzun/elips")
print("solidity    : 1 = tamamen dışbükey, düşük değer = girintili/düzensiz şekil\n")

# Ölçümlerin dağılımı
fig, ax = plt.subplots(1, 4, figsize=(18, 3.5))
ax[0].hist(hucreler.area, bins=25);                      ax[0].set_title("Alan (piksel)")
ax[1].hist(hucreler.equivalent_diameter_area, bins=25);  ax[1].set_title("Eşdeğer çap (piksel)")
ax[2].hist(hucreler.eccentricity, bins=25);              ax[2].set_title("Eccentricity")
ax[3].hist(hucreler.intensity_mean, bins=25);            ax[3].set_title("Ortalama parlaklık")
for a in ax:
    a.set_ylabel("Hücre sayısı")
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

hucreler[["area", "equivalent_diameter_area", "eccentricity", "solidity",
          "intensity_mean", "intensity_min", "intensity_max"]].describe().round(2)

In [ ]:
# Bölüm 2.5 (a): Olası güçlüklerin görsel incelenmesi
#
# Ham görüntüler çok karanlık olduğu için (piksellerin %90'ı <= 6) YALNIZCA GÖSTERİM AMACIYLA
# kontrast artırılır: [0, p99.9] aralığına kırpılır ve gamma=0.5 uygulanır (soluk bölgeler açılır).
# Bu işlem segmentasyonda kullanılmaz.
# Kare seçimi keyfi olmasın diye sekansın başı, ortası ve sonu alınır.
# Kırmızı çizgiler: 01_GT/TRA takip etiketleri (referansa göre hücrelerin yeri).
from skimage.segmentation import find_boundaries

def gosterim_icin(g, ust_yuzdelik=99.9, gamma=0.5):
    ust = np.percentile(g, ust_yuzdelik)
    return (np.clip(g, 0, ust) / ust) ** gamma

ornek_kareler = [0, 45, 91]
fig, ax = plt.subplots(2, 3, figsize=(18, 12))
for i, kare_no in enumerate(ornek_kareler):
    g = goruntu_oku(VERI / "01" / f"t{kare_no:03d}.tif")
    tra = goruntu_oku(VERI / "01_GT" / "TRA" / f"man_track{kare_no:03d}.tif")

    ax[0, i].imshow(g, cmap="gray", vmin=0, vmax=255)
    ax[0, i].set_title(f"t{kare_no:03d} - ham görüntü (0-255)")

    renkli = np.dstack([gosterim_icin(g)] * 3)
    renkli[find_boundaries(tra, mode="outer")] = [1, 0, 0]
    ax[1, i].imshow(renkli)
    ax[1, i].set_title(f"t{kare_no:03d} - gösterim için kontrast + TRA ({len(np.unique(tra)) - 1} hücre)")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Bölüm 2.5 (b): Güçlüklerin sayısal olarak ölçülmesi (Sekans 01, 92 karenin tamamı)
#
# 1) Temas eden / çok yakın hücreler : TRA etiketleri 10 piksel genişletilince birleşen hücre sayısı
#    (TRA etiketleri gerçek hücreden biraz küçük olduğu için genişletme gerçek sınıra yaklaştırır)
# 2) Kontrast yetersizliği             : ortalama parlaklığı arka plana yakın olan hücrelerin oranı (Bölüm 2.4)
# 3) Gürültü                           : hücrelerin dışında kalan küçük parlak lekeler (>= 50 yoğunluk, < 100 piksel)
# 4) Düzensiz arka plan                : hücre dışı piksellerin 8x8 bloklardaki ortalaması
cekirdek = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))   # yarıçap 10 piksel
kare_satirlari = []
blok_haritalari = []
for kare_no in range(92):
    g = goruntu_oku(VERI / "01" / f"t{kare_no:03d}.tif")
    tra = goruntu_oku(VERI / "01_GT" / "TRA" / f"man_track{kare_no:03d}.tif")
    hucre_sayisi = len(np.unique(tra)) - 1

    genis = cv2.dilate((tra > 0).astype(np.uint8), cekirdek)
    bilesen_sayisi = cv2.connectedComponents(genis)[0] - 1          # 0 = arka plan

    hucre_disi = cv2.dilate(genis, cekirdek) == 0                   # hücrelerden emin olunacak kadar uzak bölge
    parlak = ((g >= 50) & hucre_disi).astype(np.uint8)
    n, _, stats, _ = cv2.connectedComponentsWithStats(parlak)
    kucuk_leke = int(np.sum(stats[1:, cv2.CC_STAT_AREA] < 100))

    bloklar = np.full((8, 8), np.nan)
    for r in range(8):
        for c in range(8):
            dilim = (slice(r * 128, (r + 1) * 128), slice(c * 128, (c + 1) * 128))
            if hucre_disi[dilim].sum() > 1000:
                bloklar[r, c] = g[dilim][hucre_disi[dilim]].mean()
    blok_haritalari.append(bloklar)

    kare_satirlari.append({"kare": kare_no, "hucre": hucre_sayisi,
                           "birlesen_hucre": hucre_sayisi - bilesen_sayisi,
                           "kucuk_parlak_leke": kucuk_leke,
                           "arka_plan_blok_min": np.nanmin(bloklar),
                           "arka_plan_blok_max": np.nanmax(bloklar)})

guclukler = pd.DataFrame(kare_satirlari)

print("1) Temas eden / çok yakın hücreler")
print(f"   Karelerin %{(guclukler.birlesen_hucre > 0).mean()*100:.0f}'inde en az bir birleşme var; "
      f"kare başına ortalama {guclukler.birlesen_hucre.mean():.2f}, en fazla {guclukler.birlesen_hucre.max()}")
print("2) Kontrast yetersizliği (SEG hücreleri)")
print(f"   Ortalama parlaklığı <= 6 olan (arka planın %90'ıyla aynı düzeyde) hücreler: "
      f"%{(hucreler.intensity_mean <= 6).mean()*100:.0f}; <= 10 olanlar: %{(hucreler.intensity_mean <= 10).mean()*100:.0f}")
print("3) Gürültü")
print(f"   Hücre dışında kare başına ortalama {guclukler.kucuk_parlak_leke.mean():.1f} küçük parlak leke "
      f"(en fazla {guclukler.kucuk_parlak_leke.max()})")
print("4) Düzensiz arka plan")
print(f"   Hücre dışı 8x8 blok ortalamaları: en düşük {guclukler.arka_plan_blok_min.mean():.2f}, "
      f"en yüksek {guclukler.arka_plan_blok_max.mean():.2f} (92 karenin ortalaması)")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(guclukler.kare, guclukler.hucre, label="Referans hücre sayısı (TRA)")
ax[0].plot(guclukler.kare, guclukler.hucre - guclukler.birlesen_hucre, label="Genişletince ayrı kalan bileşen")
ax[0].set_xlabel("Kare numarası"); ax[0].set_title("Temas eden hücrelerin sayıma etkisi")
ax[0].legend(); ax[0].grid(alpha=0.3)
ort_harita = np.nanmean(blok_haritalari, axis=0)
im = ax[1].imshow(ort_harita, cmap="viridis")
ax[1].set_title("Hücre dışı arka plan - 8x8 blok ortalaması (92 kare)")
plt.colorbar(im, ax=ax[1])
plt.tight_layout()
plt.show()